### Load data

In [ ]:
import pandas as pd
import os
from pathlib import Path

project_dir = Path.cwd().parent.parent

questions = pd.read_csv(os.path.join(project_dir, "data/raw_datasets/questions.csv"))   # Data with raw parameters info
instructions = pd.read_csv(os.path.join(project_dir, "data/instruction_sets/instructions.csv"))   # Data in instructions style (instruction converted from parameters)

In [5]:
len(questions), len(instructions)

(8312, 8312)

#### Check if two data are match

In [3]:
# Check if they are match

for i in range(len(instructions)):
    q_exercise_types = questions.iloc[i]["exercise_type"].split(",")
    # print(q_exercise_types)
    [t.lower() in instructions.iloc[i]["Input"].lower() for t in q_exercise_types]
    if all(t.lower() in instructions.iloc[i]["Input"].lower() for t in q_exercise_types):
        # print(True, i)
        pass
    else:
        print(False, i)
    # break

### <Method 1> Manually change goundtruth params to have new instruction that mismatch the goundthuth gneration

In this way we label the level of mismatch of each param as adherence score

#### Sameple subset of questions and add instructions

In [ ]:
# set random seed
import random
random.seed(42)

questions_subset = questions.sample(60)
# rename column "content" to "generation"
questions_subset.rename(columns={"content": "generation"}, inplace=True)
# copy column "Input" from instructions as "instruction" in questions_subset
questions_subset["instruction"] = instructions["Input"]
# move column "instruction" in front of "generation"
questions_subset.insert(0, "instruction", questions_subset.pop("instruction"))
questions_subset.to_csv(os.path.join(project_dir, "data/complete_data_subset-60.csv"), index=False)
questions_subset

#### Annotation

Use `Content-Generation/eval/adherence/annotation_demo.py` for annotatin, resulting in `Content-Generation/eval/adherence/annotated_dataset.csv`

In [2]:
annotated_dataset = pd.read_csv("annotated_dataset.csv")
annotated_dataset.head()
len(annotated_dataset)
annotated_dataset = annotated_dataset[5:20]
annotated_dataset

,original_id,original_instruction,original_subject,original_exercise_type,original_difficulty,original_grade_level,original_modifier,original_generation,new_instruction,new_subject,new_exercise_type,new_difficulty,new_grade_level,new_modifier,adherence_subject,adherence_exercise_type,adherence_difficulty,adherence_grade_level,adherence_modifier,annotation_timestamp
5,35,Give me a combination of a word problem and an...,Mathematics,"Word Problem, Arithmetic Exercise",D,5,"With Answer, Exercise with division",Sofia has \( \frac{3}{4} \) of a cake. She div...,Give me an arithmetic exercise for a 5th grade...,Mathematics,Arithmetic Exercise,D,5,"With Hint, Exercise on long division",1,2.0,0.0,4.0,1.0,2025-06-28T10:18:55.493621
6,36,Give me a drawing exercise for a 1st grade stu...,Mathematics,Drawing Exercise,D,1,Exercise on asymmetrical shapes,Draw 2 shapes that are asymmetrical,Give me a multiple choice for a 1st grade stud...,Arts,Multiple choice,D,1,Exercise on symmetrical shapes,1,0.0,0.0,1.0,3.0,2025-06-28T10:21:12.803828
7,37,Give me a short answer exercise for a 4th grad...,Computer Science,Short Answer Exercise,D,4,Multi-part,\begin{enumerate}\n\item What shape of box is ...,Give me a short answer exercise for a 4th grad...,Computer Science,Short Answer Exercise,D,4,"Multi-part, Exercise on class diagrams",1,2.0,0.0,4.0,2.0,2025-06-28T10:24:01.883738
8,38,Give me a combination of a word problem and an...,Mathematics,"Word Problem, Arithmetic Exercise",D,6,Exercise on speed comparison,Oliver and Paul started from Town M towards To...,Give me a combination of a word problem and an...,Mathematics,"Word Problem, Arithmetic Exercise",D,7,Exercise on units of measurement,1,4.0,0.0,2.0,1.0,2025-06-28T10:26:01.105905
9,39,Give me a combination of a punctuation exercis...,English,"Punctuation Exercise, Writing Exercise",D,3,Exercise on direct speech sentences,Copy and punctuate these direct speech sentenc...,Give me a combination of a punctuation exercis...,German,"Punctuation Exercise, Writing Exercise",D,3,Exercise on direct speech sentences,0,4.0,0.0,4.0,4.0,2025-06-28T10:26:51.979141
10,40,Give me a combination of a group activity and ...,History,"Group Activity, Research Exercise",D,3,Exercise on historical events in media,Work in a group. Find out when:\n\begin{enumer...,Give me a combination of a group activity and ...,Social Science,"Group Activity, Research Exercise",D,3,Exercise on nationalist movements,0,4.0,0.0,3.0,1.0,2025-06-28T10:28:25.011222
11,41,Give me a combination of a group activity and ...,Social Studies,"Group Activity, Research Exercise",D,4,Exercise on recent public information campaigns,Work in a group to find out about some recent ...,Give me a combination of an essay and a long a...,Social Studies,"Essay, Long answer",D,8,Exercise on recent public information campaigns,1,2.0,0.0,1.0,4.0,2025-06-28T10:29:56.191315
12,42,Give me a word scramble exercise for a student...,Dutch,Word Scramble Exercise,D,G,"With Answer, With Hint",All the letters of these words are jumbled up....,Give me a combination of a word scramble exerc...,Spanish,"Word Scramble Exercise, Translation Exercise",D,G,"With Answer, With Hint",0,1.0,0.0,0.0,4.0,2025-06-28T10:31:29.947448
13,43,Give me a multiple choice exercise for a 1st g...,History,Multiple Choice Exercise,D,1,M,Edmund Hillary was the first person to climb:\...,Give me a true or false for a 1st grade studen...,Geography,True or False,D,1,M,0,1.0,0.0,4.0,0.0,2025-06-28T10:32:37.221027
14,44,Give me a fill in the blank exercise for a 6th...,Mathematics,Fill in the Blank Exercise,D,6,Exercise on sentence completion,Complete these sentences: A shape is a polygon...,Give me a fill in the blank exercise for a 9th...,English,Fill in the Blank Exercise,D,9,Exercise on sentence completion,0,0.0,0.0,2.0,3.0,2025-06-28T10:34:26.009770


In [3]:
# Mask adherence_<param> to None when corresponding new_<param> flag is in [S, G, D, M, T]
annotated_dataset.loc[annotated_dataset['new_subject'].isin(['S', 's']), 'adherence_subject'] = None
annotated_dataset.loc[annotated_dataset['new_exercise_type'].isin(['T', 't']), 'adherence_exercise_type'] = None
annotated_dataset.loc[annotated_dataset['new_grade_level'].isin(['G', 'g']), 'adherence_grade_level'] = None
annotated_dataset.loc[annotated_dataset['new_modifier'].isin(['M', 'm']), 'adherence_modifier'] = None
annotated_dataset.loc[annotated_dataset['new_difficulty'].isin(['D', 'd']), 'adherence_difficulty'] = None

#### Run adherence evaluation

In [4]:
from adherence_eval import adherence_score


# apply adherence_score func to each row to take new_instruction and original_generation as input and return values to 4 new columns, the return of adherence_score is {"Subject score": x, "Exercise type score": x, "Grade level score": x, "Modification score": x}

# Apply the adherence_score function row-wise and expand the results into new columns
scores = annotated_dataset.apply(
    lambda row: adherence_score(row['new_instruction'], row['original_generation']),
    axis=1
)
# Convert list of dicts into a DataFrame
scores_df = pd.DataFrame(scores.tolist())
# Assign the four perspective scores to new columns
annotated_dataset[['subject_score',
                   'exercise_type_score',
                   'grade_level_score',
                   'modifier_score']] = scores_df[[
                       'subject_score',
                       'exercise_type_score',
                       'grade_level_score',
                       'modifier_score'
                   ]]
# Optionally display the updated DataFrame
annotated_dataset.head()

Using structured output
{'subject_score': 1, 'exercise_type_score': 1, 'grade_level_score': 4, 'modifier_score': 2}
Using structured output
{'subject_score': 1, 'exercise_type_score': 0, 'grade_level_score': 1, 'modifier_score': 3}
Using structured output
{'subject_score': 1, 'exercise_type_score': 4, 'grade_level_score': 4, 'modifier_score': 4}
Using structured output
{'subject_score': 1, 'exercise_type_score': 4, 'grade_level_score': 4, 'modifier_score': 4}
Using structured output
{'subject_score': 1, 'exercise_type_score': 4, 'grade_level_score': 4, 'modifier_score': 4}
Using structured output
{'subject_score': 1, 'exercise_type_score': 4, 'grade_level_score': 4, 'modifier_score': 1}
Using structured output
{'subject_score': 1, 'exercise_type_score': 2, 'grade_level_score': 4, 'modifier_score': 4}
Using structured output
{'subject_score': 1, 'exercise_type_score': 4, 'grade_level_score': 3, 'modifier_score': 4}
Using structured output
{'subject_score': 1, 'exercise_type_score': 0, '

,original_id,original_instruction,original_subject,original_exercise_type,original_difficulty,original_grade_level,original_modifier,original_generation,new_instruction,new_subject,...,adherence_subject,adherence_exercise_type,adherence_difficulty,adherence_grade_level,adherence_modifier,annotation_timestamp,subject_score,exercise_type_score,grade_level_score,modifier_score
5,35,Give me a combination of a word problem and an...,Mathematics,"Word Problem, Arithmetic Exercise",D,5,"With Answer, Exercise with division",Sofia has \( \frac{3}{4} \) of a cake. She div...,Give me an arithmetic exercise for a 5th grade...,Mathematics,...,1.0,2.0,NaN,4.0,1.0,2025-06-28T10:18:55.493621,1.0,4.0,4.0,1.0
6,36,Give me a drawing exercise for a 1st grade stu...,Mathematics,Drawing Exercise,D,1,Exercise on asymmetrical shapes,Draw 2 shapes that are asymmetrical,Give me a multiple choice for a 1st grade stud...,Arts,...,1.0,0.0,NaN,1.0,3.0,2025-06-28T10:21:12.803828,1.0,2.0,4.0,4.0
7,37,Give me a short answer exercise for a 4th grad...,Computer Science,Short Answer Exercise,D,4,Multi-part,\begin{enumerate}\n\item What shape of box is ...,Give me a short answer exercise for a 4th grad...,Computer Science,...,1.0,2.0,NaN,4.0,2.0,2025-06-28T10:24:01.883738,1.0,4.0,3.0,4.0
8,38,Give me a combination of a word problem and an...,Mathematics,"Word Problem, Arithmetic Exercise",D,6,Exercise on speed comparison,Oliver and Paul started from Town M towards To...,Give me a combination of a word problem and an...,Mathematics,...,1.0,4.0,NaN,2.0,1.0,2025-06-28T10:26:01.105905,1.0,0.0,1.0,0.0
9,39,Give me a combination of a punctuation exercis...,English,"Punctuation Exercise, Writing Exercise",D,3,Exercise on direct speech sentences,Copy and punctuate these direct speech sentenc...,Give me a combination of a punctuation exercis...,German,...,0.0,4.0,NaN,4.0,4.0,2025-06-28T10:26:51.979141,1.0,4.0,4.0,4.0


In [5]:
from adherence_reliability import (
    evaluate_ordinal_adherence_reliability, 
    evaluate_binary_adherence_reliability
)

# Compute binary adherence reliability
binary_valid = annotated_dataset[['subject_score', 'adherence_subject']].dropna()
evaluate_binary_adherence_reliability(binary_valid['subject_score'], binary_valid['adherence_subject'])

{'Accuracy': 0.6,
 'Cohen_Kappa': 0.19999999999999996,
 'F1_Score': 0.7142857142857143,
 'MCC': 0.3333333333333333,
 'Precision': 1.0,
 'Recall': 0.5555555555555556,
 'Confusion_Matrix': array([[1, 0],
        [4, 5]])}

In [6]:
metrics = [
    ('exercise_type_score', 'adherence_exercise_type'),
    ('grade_level_score',    'adherence_grade_level'),
    ('modifier_score',       'adherence_modifier'),
]

# Compute ordinal adherence reliability
for pred_col, true_col in metrics:
    ordinal_valid = annotated_dataset[[pred_col, true_col]].dropna()
    result = evaluate_ordinal_adherence_reliability(ordinal_valid[true_col], ordinal_valid[pred_col])
    print(f"Ordinal adherence reliability for {pred_col} vs {true_col}:")
    print("="*50)
    for k, v in result.items():
        print(f"{k}: {v}")
    print("\n")
    

Ordinal adherence reliability for exercise_type_score vs adherence_exercise_type:
MAE: 2.0
Quadratic_Weighted_Kappa: -0.17647058823529416
Pearson_Correlation: -0.2032789070454354
Spearman_Correlation: -0.058760596034724104


Ordinal adherence reliability for grade_level_score vs adherence_grade_level:
MAE: 1.1111111111111112
Quadratic_Weighted_Kappa: 0.06896551724137934
Pearson_Correlation: 0.08419653359482283
Spearman_Correlation: -0.0367465459870082


Ordinal adherence reliability for modifier_score vs adherence_modifier:
MAE: 1.2222222222222223
Quadratic_Weighted_Kappa: 0.3440233236151603
Pearson_Correlation: 0.36334017132308305
Spearman_Correlation: 0.3417533713006601




### <Method 2> Sample binary classification dataset to test adherence evaluator
#### Create coherence score (0-4) test dataset

In [ ]:
"""
Define sample function.

Sample two random (params,excercise) instance, use the second exercise as the generated excerise of the first params
The match params are score of coherence.
"""
import numpy as np
import random

# Define function to sample and calculate matches
def sample_and_match(df):
    """
    Sample two (can be the same) row and calculate match params as score
    """
    # With 0.2 probability take the input output from the same sample to have score of 4
    if random.random() < 0.2:
        # Sample the same row
        sampled_rows = df.sample(1)
        # Use the same index
        idx1 = idx2 = sampled_rows.index[0]
    else:
        # Sample 2 rows
        sampled_rows = df.sample(2)
        # Extract the indices of the sampled rows
        idx1, idx2 = sampled_rows.index
        
    # Exclude "content" and "difficulty" column from the comparison
    cols_to_compare = df.columns.difference(['content', 'difficulty'])

    # Count the number of matching columns (excluding "content" and "difficulty")
    match = df.iloc[idx1][cols_to_compare] == df.iloc[idx2][cols_to_compare]
    num_matched_columns = np.sum(match)
    
    # Create a result dictionary
    result = {
        'score': num_matched_columns,
        'index1': idx1,
        'index2': idx2
    } | match.astype(int).to_dict() # Merge two dictionaries
    
    return result

# Example usage
result_df = sample_and_match(questions)
print(result_df)

{'score': 1, 'index1': 6640, 'index2': 7134, 'exercise_type': 0, 'grade_level': 0, 'modifier': 0, 'subject': 1}


In [30]:
questions.iloc[result_df["index1"]], questions.iloc[result_df["index2"]]

(subject                               Mathematics
 exercise_type                    Drawing Exercise
 difficulty                                      D
 grade_level                                     2
 modifier                                        M
 content          Draw a right angle and label it.
 Name: 8253, dtype: object,
 subject                               Mathematics
 exercise_type                    Drawing Exercise
 difficulty                                      D
 grade_level                                     2
 modifier                                        M
 content          Draw a right angle and label it.
 Name: 8253, dtype: object)

In [ ]:
""" 
Generate a coherence score sample pool which should be big enough for later even distribution sampling
"""

# Generate a sample pool of size 20000
results = []
for _ in range(20000):
    result = sample_and_match(questions)
    results.append(result)

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Display the resulting DataFrame
# print(results_df)

# Display the distribution of different values of num_matched_column
distribution = results_df['score'].value_counts().sort_index()

# Display the distribution
print(distribution)

score
0    10506
1     4363
2      981
3      110
4     4040
Name: count, dtype: int64


In [ ]:
"""
Create even distribution coherence score dataset with each score having 100 instances
"""

# Sample 50 values from each category of score
def sample_from_each_category(df, column, sample_size=50):
    # Group by the specified column (score)
    grouped = df.groupby(column)
    
    # Sample up to 'sample_size' rows from each group
    sampled_df = grouped.apply(lambda x: x.sample(min(len(x), sample_size), replace=False)).reset_index(drop=True)
    
    return sampled_df

# Apply the function to sample 50 rows from each category of score (0-4)
sampled_results_df = sample_from_each_category(results_df, 'score', sample_size=100)

# Display the new DataFrame with the sampled results
print(sampled_results_df['score'].value_counts().sort_index())

score
0    100
1    100
2    100
3    100
4    100
Name: count, dtype: int64


In [ ]:
# sampled_results_df.to_csv("data/coherence_eval.csv", index=False)

### Evaluate the method `coherence evaluation by llm`

In [ ]:
sampled_results_df = pd.read_csv("data/coherence_eval.csv")

In [15]:
# Sample "Input" based on row1.index and "Output" based on row2.index
sampled_results_df['Input'] = sampled_results_df['index1'].apply(lambda idx: instructions.loc[idx, 'Input'])
sampled_results_df['Output'] = sampled_results_df['index2'].apply(lambda idx: instructions.loc[idx, 'Output'])
sampled_results_df['OriInput'] = sampled_results_df['index2'].apply(lambda idx: instructions.loc[idx, 'Input'])
sampled_results_df['OriOutput'] = sampled_results_df['index1'].apply(lambda idx: instructions.loc[idx, 'Output'])

In [16]:
shuffle_data = sampled_results_df.sample(frac=1, ignore_index=True)

In [ ]:
"""
Define llm calling function
"""

from openai import OpenAI
from dotenv import load_dotenv

# Set up your OpenAI API key
load_dotenv() 
client = OpenAI()

# Text generation using GPT-4
def generate_text(sys_prompt, user_prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o",  # Use "gpt-4-turbo" for a faster, cheaper alternative
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=150,  # Limit the number of tokens in the output
            temperature=0.7,  # Adjust the creativity level of the model
            n=1,  # Number of completions to generate
            stop=None  # Stop generation based on special characters or tokens
        )
        return response.choices[0].message.content
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# # Example usage
# prompt = "Write a short story about a robot learning to play the piano."
# generated_text = generate_text(prompt)
# print(generated_text)

In [ ]:
"""
Define coherence score function
"""
import re

system_prompt = """
Role:
You are an exercise generation evaluator. 

Task:
Given an query-answer pair, where the query is a demand and the answer is the generated exercise,
you will score the coherence of the excercise from 4 perspectives:
- Subject: score 1 if the subject is correct else 0.
- Exercise type: score 1 if the exercise type is correct else 0.
- Grade level: score 1 if the grade level is close to the given according to 1-12 Amecian grade system else 0.
- Modification: score 1 if the specific details matches the requirement else 0.
Don't be too strict in each perspective, you can give them score as long as it is fulfilled in a cetain level. Your final score is the sum of all these four scores.

Return:
Provide your answer in the following format:
{"Subject score": x, "Exercise type score": x, "Grade level score": x, "Modification score": x, "Total score": x}"""

def parse_response(response):
    response = re.findall(r"{[\s\S]+}", response)[0]
    return eval(response)

result_list = []

def coherence_eval(x):
    input, output = x['Input'], x['Output']
    user_prompt = str({"Query": input, "Answer": output})
    try:
        response = generate_text(system_prompt, user_prompt)
        result = parse_response(response)
    except Exception as e:
        print(e)
        print(response)
        return {'Subject score': None, 'Exercise type score': None, 'Grade level score': None, 'Modification score': None, 'Total score': None}

    return result

In [ ]:
sampled_results_df = shuffle_data[:100]

In [ ]:
# """
# Predict coherence scores (by for foop)
# """

# predict_scores = {}
# for index, row in sampled_results_df.iterrows():
#     input, output = row['Input'], row['Output']
#     user_prompt = f"Query: \"{input}\"\n\nAnswer: \"{output}\""
#     response = generate_text(system_prompt, user_prompt)
#     result = parse_response(response)
#     predict_scores[index] = result["Total score"]
#     break

In [ ]:
"""
Predict coherence scores
"""

sampled_results_df[['Subject score', 'Exercise type score', 'Grade level score', 'Modification score', 'Total score']] = \
    sampled_results_df.apply(coherence_eval, axis=1, result_type='expand')

/tmp/ipykernel_17520/3161120063.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sampled_results_df[['Subject score', 'Exercise type score', 'Grade level score', 'Modification score', 'Total score']] = \


In [33]:
sampled_results_df.to_csv('data/coherence_eval_result_2.csv', index=False)

In [ ]:
"""
Reload coherence score prediction and calculate accuracy
"""

import numpy as np
import pandas as pd

data = pd.read_csv('data/coherence_eval_result_2.csv')

subject_correct = sum(data['subject'] == data['Subject score']) / len(data)
type_correct = sum(data['exercise_type'] == data['Exercise type score']) / len(data)
grade_correct = sum(data['grade_level'] == data['Grade level score']) / len(data)
modifier_correct = sum(data['modifier'] == data['Modification score']) / len(data)
mAA = np.mean([subject_correct, type_correct, grade_correct, modifier_correct])

print(f"Evaluate by llm accuracy:\nSubject: {subject_correct},\nExercise type: {type_correct},\nGrade level: {grade_correct},\
      \nModifier: {modifier_correct},\nMean Average Accuracy: {mAA}")

In [ ]:
# 0: (0.73, 0.79, 0.78, 0.83, 0.7825)
# 2: (0.76, 0.77, 0.81, 0.85, 0.7975)